In [1]:
import torch
import torch.nn  as nn
import numpy as np 
import pandas as pd
from sklearn.preprocessing  import StandardScaler 
 
# 参数配置 
config = {
    'seq_length': 30,      # 时间窗口长度
    'feature_size': 8,     # 特征维度(收盘价、开盘价、高低价、成交量+技术指标)
    'd_model': 64,         # 模型维度 
    'nhead': 4,            # 注意力头数 
    'num_layers': 3,       # Transformer编码器层数 
    'dropout': 0.1
}
 
class StockTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embedding  = nn.Linear(config['feature_size'], config['d_model'])
        self.pos_encoder  = PositionalEncoding(config['d_model'], config['dropout'])
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=config['d_model'],
            nhead=config['nhead'],
            dropout=config['dropout']
        )
        self.transformer_encoder  = nn.TransformerEncoder(encoder_layers, config['num_layers'])
        
    def forward(self, src):
        # src形状: [seq_len, batch_size, feature_size]
        src = self.embedding(src)  * np.sqrt(config['d_model']) 
        src = self.pos_encoder(src) 
        output = self.transformer_encoder(src) 
        return output  # 输出形状: [seq_len, batch_size, d_model]
 
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout  = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1) 
        div_term = torch.exp(torch.arange(0,  d_model, 2) * (-np.log(10000.0)  / d_model))
        pe = torch.zeros(max_len,  1, d_model)
        pe[:, 0, 0::2] = torch.sin(position  * div_term)
        pe[:, 0, 1::2] = torch.cos(position  * div_term)
        self.register_buffer('pe',  pe)
 
    def forward(self, x):
        x = x + self.pe[:x.size(0)] 
        return self.dropout(x) 
 
# 数据预处理流程 
def preprocess_data(data_path):
    # 加载股票数据（示例格式）
    df = pd.read_csv(data_path,  parse_dates=['Date'])
    
    # 特征工程：添加技术指标
    df['MA5'] = df['Close'].rolling(5).mean()
    df['MA20'] = df['Close'].rolling(20).mean()
    df['Return'] = df['Close'].pct_change()
    df['Volatility'] = df['Return'].rolling(5).std()
    df.dropna(inplace=True) 
    
    # 选择特征列 
    features = ['Open', 'High', 'Low', 'Close', 'Volume', 'MA5', 'MA20', 'Volatility']
    data = df[features].values 
    
    # 标准化 
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data) 
    
    # 创建时间序列样本 
    X = []
    for i in range(len(scaled_data) - config['seq_length']):
        X.append(scaled_data[i:i+config['seq_length']]) 
    return torch.tensor(np.array(X),  dtype=torch.float32) 
 
# 特征提取函数
def extract_features(model, data_loader):
    model.eval() 
    features = []
    with torch.no_grad(): 
        for batch in data_loader:
            src = batch.permute(1,  0, 2)  # [seq_len, batch_size, feature_size]
            output = model(src)
            # 取最后一个时间步的特征作为序列表示
            seq_features = output[-1, :, :].cpu().numpy()
            features.extend(seq_features) 
    return np.array(features) 
 
# 使用示例 
if __name__ == "__main__":
    # 数据准备 
    stock_data = preprocess_data('stock_data.csv') 
    data_loader = torch.utils.data.DataLoader(stock_data,  batch_size=32, shuffle=True)
    
    # 初始化模型 
    model = StockTransformer(config)
    
    # 特征提取
    stock_features = extract_features(model, data_loader)
    print(f"提取的特征矩阵形状：{stock_features.shape}") 

FileNotFoundError: [Errno 2] No such file or directory: 'stock_data.csv'

In [ ]:
import torch
import numpy as np 
import pandas as pd
import xgboost as xgb
from sklearn.model_selection  import TimeSeriesSplit 
from torch import nn
from sklearn.preprocessing  import RobustScaler 
from sklearn.metrics  import mean_absolute_percentage_error 
 
# 配置参数
CONFIG = {
    'seq_len': 60,          # 时间序列窗口 
    'transformer': {
        'd_model': 128,       
        'nhead': 8,
        'num_layers': 4,
        'dropout': 0.2
    },
    'xgb_params': {
        'max_depth': 7,
        'learning_rate': 0.02,
        'n_estimators': 1500,
        'subsample': 0.8
    }
}
 
class HybridModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.transformer  = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=CONFIG['transformer']['d_model'],
                nhead=CONFIG['transformer']['nhead'],
                dropout=CONFIG['transformer']['dropout']
            ), 
            num_layers=CONFIG['transformer']['num_layers']
        )
        self.feature_proj  = nn.Linear(8, CONFIG['transformer']['d_model'])
        
    def forward(self, ts_input):
        # 时序特征提取 
        ts_features = self.feature_proj(ts_input) 
        ts_features = ts_features.permute(1,  0, 2)
        transformer_out = self.transformer(ts_features) 
        return transformer_out[-1].squeeze()
 
 
class MarketDataset(torch.utils.data.Dataset): 
    def __init__(self, stock_path, macro_path):
        # 加载股票数据与宏观数据
        stock_df = self._process_stock(stock_path)
        macro_df = self._process_macro(macro_path)
        
        # 对齐时间轴 
        self.full_data  = pd.merge_asof(stock_df.sort_index(),  
                                     macro_df.sort_index(), 
                                     left_index=True, 
                                     right_index=True)
        
        # 特征工程 
        self.scaler  = RobustScaler()
        self.ts_features  = self.scaler.fit_transform( 
            self.full_data[['open',  'high', 'low', 'close', 'volume', 
                           'macd', 'rsi', 'atr']].values)
        
        # 外部特征
        self.external_features  = self.full_data[['cpi',  'interest_rate',
                                                'industry_index', 
                                                'sentiment_score']].values
        
        self.targets  = self.full_data['close'].pct_change(24).shift(-24).values[24:-24] 
        
    def __getitem__(self, idx):
        ts_window = self.ts_features[idx:idx+CONFIG['seq_len']] 
        external = self.external_features[idx+CONFIG['seq_len']] 
        target = self.targets[idx+CONFIG['seq_len']] 
        return (torch.FloatTensor(ts_window),
                torch.FloatTensor(external),
                torch.FloatTensor([target]))
    
    def _add_technical_features(self, df):
        # 添加MACD、RSI、ATR等技术指标
        ...
        return df 
    
    def _process_macro(self, path):
        # 宏观经济数据处理 
        ...
        return df 
 
# 两阶段训练流程
def train_pipeline():
    # 第一阶段：训练Transformer特征提取器 
    trans_model = HybridModel()
    trans_optim = torch.optim.AdamW(trans_model.parameters(),  lr=1e-4)
    
    # 时序特征预训练
    for epoch in range(10):
        for ts, _, target in train_loader:
            pred = trans_model(ts)
            loss = nn.MSELoss()(pred, target)
            loss.backward() 
            trans_optim.step() 
    
    # 第二阶段：XGBoost集成训练
    trans_model.eval() 
    train_features = []
    
    with torch.no_grad(): 
        for ts, ext, _ in train_loader:
            ts_feat = trans_model(ts).numpy()
            combined_feat = np.concatenate([ts_feat,  ext.numpy()],  axis=1)
            train_features.append(combined_feat) 
            
    X_train = np.vstack(train_features) 
    y_train = dataset.targets[CONFIG['seq_len']:len(train_features)+CONFIG['seq_len']] 
    
    xgb_model = xgb.XGBRegressor(**CONFIG['xgb_params'])
    xgb_model.fit(X_train,  y_train, 
                 eval_set=[(X_val, y_val)],
                 early_stopping_rounds=50)
    
    return trans_model, xgb_model
 
# 预测模块 
def predict_next_day(model_pair, latest_data):
    trans_model, xgb_model = model_pair 
    ts_tensor = torch.FloatTensor(latest_data['ts_window'])
    ext_tensor = torch.FloatTensor(latest_data['external'])
    
    with torch.no_grad(): 
        ts_feat = trans_model(ts_tensor.unsqueeze(0)).numpy() 
        
    combined_feat = np.concatenate([ts_feat,  ext_tensor.numpy().reshape(1,-1)],  axis=1)
    return xgb_model.predict(combined_feat)[0] 